# SmolVLA Deployment Simulation

**Lecture 3, Part 4 — Vizuara Modern Robot Learning Bootcamp**

In this notebook, we simulate the **full deployment pipeline** for SmolVLA on a real SO-101 arm — from dataset exploration to voice-controlled task switching.

| Section | Topic | Key Idea |
|---------|-------|----------|
| Step 1 | Dataset Exploration | Load and inspect a real LeRobot dataset |
| Step 2 | State Vector | Visualize 6-DOF joint trajectories |
| Step 3 | Camera Views | Preprocessing and rename maps |
| Step 4 | Action Chunking | Deque-based efficient inference |
| Step 5 | Voice Control | Thread-safe task switching |
| Step 6 | Intent Classification | Semantic color recognition with sentence-transformers |
| Step 7 | Training Pipeline | Configuration and GPU options |
| Step 8 | predict_action vs sample_actions | The right vs wrong inference loop |
| Step 9 | Full Pipeline | End-to-end summary and best practices |

By the end, you'll understand every piece of the deployment puzzle — **without needing a physical robot**.

---

In [ ]:
# Install dependencies (Colab has most of these, but just in case)
!pip install -q torch numpy matplotlib datasets huggingface_hub sentence-transformers Pillow 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque, OrderedDict
import threading
import time

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor'] = BG
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

---
## Step 1: Setup + LeRobot Dataset Exploration

We'll load a real SO-101 dataset from HuggingFace. If the dataset is unavailable (private or network issues), we generate a **synthetic fallback** with the exact same structure.

In [ ]:
# Try loading the real dataset; fall back to synthetic if unavailable
USE_REAL_DATA = False
try:
    from datasets import load_dataset
    ds = load_dataset("RajatDandekar/so101_box_to_bowl", split="train")
    USE_REAL_DATA = True
    print(f"Loaded real dataset: {len(ds)} samples")
except Exception as e:
    print(f"Could not load HF dataset: {e}")
    print("Generating synthetic fallback dataset...\n")

# ---- Synthetic fallback ----
if not USE_REAL_DATA:
    JOINT_NAMES = ['shoulder_pan', 'shoulder_lift', 'elbow_flex',
                   'wrist_flex', 'wrist_roll', 'gripper']
    COLORS = ['red', 'green', 'blue']
    EPISODES_PER_COLOR = 15
    STEPS_PER_EPISODE = 90

    synthetic_data = {
        'observation.state': [],
        'action': [],
        'task': [],
        'episode_index': [],
        'frame_index': [],
    }

    episode_idx = 0
    for color in COLORS:
        for ep in range(EPISODES_PER_COLOR):
            # Create smooth sinusoidal trajectories with per-episode variation
            t = np.linspace(0, 2 * np.pi, STEPS_PER_EPISODE)
            phase_offsets = np.random.uniform(0, np.pi, 6)
            amplitudes = np.array([0.8, 1.2, 0.6, 0.4, 0.3, 1.0]) + np.random.randn(6) * 0.1
            frequencies = np.array([1.0, 0.8, 1.5, 1.2, 0.9, 0.5]) + np.random.randn(6) * 0.05

            for step in range(STEPS_PER_EPISODE):
                # Observation state: current joint positions
                state = [float(amplitudes[j] * np.sin(frequencies[j] * t[step] + phase_offsets[j]))
                         for j in range(6)]
                # Action: slightly ahead in phase (actions lead observations)
                lookahead = min(step + 2, STEPS_PER_EPISODE - 1)
                action = [float(amplitudes[j] * np.sin(frequencies[j] * t[lookahead] + phase_offsets[j])
                          + np.random.randn() * 0.02)
                          for j in range(6)]

                synthetic_data['observation.state'].append(state)
                synthetic_data['action'].append(action)
                synthetic_data['task'].append(f"Pick up the box and place it in the {color} bowl")
                synthetic_data['episode_index'].append(episode_idx)
                synthetic_data['frame_index'].append(step)

            episode_idx += 1

    # Convert to a simple dict-based dataset for uniform access
    class SyntheticDataset:
        def __init__(self, data):
            self.data = data
            self._len = len(data['task'])
        def __len__(self):
            return self._len
        def __getitem__(self, idx):
            return {k: v[idx] for k, v in self.data.items()}
        @property
        def column_names(self):
            return list(self.data.keys())

    ds = SyntheticDataset(synthetic_data)
    print(f"Generated synthetic dataset: {len(ds)} samples")
    print(f"  {EPISODES_PER_COLOR * len(COLORS)} episodes x {STEPS_PER_EPISODE} steps")

In [ ]:
# Explore dataset structure
print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)
print(f"\nColumn names: {ds.column_names}")
print(f"Total samples: {len(ds)}")

# Show shapes and dtypes
sample = ds[0]
print(f"\nSample 0:")
for key in ds.column_names:
    val = sample[key]
    if isinstance(val, list):
        print(f"  {key:30s} | shape=[{len(val)}]  dtype=float")
    else:
        print(f"  {key:30s} | type={type(val).__name__}  value={val}")

# Show sample task strings
print(f"\nSample task strings:")
seen_tasks = set()
for i in range(len(ds)):
    task = ds[i]['task']
    if task not in seen_tasks:
        seen_tasks.add(task)
        print(f"  [{i:4d}] \"{task}\"")
    if len(seen_tasks) >= 3:
        break

# Show sample observation.state and action values
print(f"\nSample observation.state values:")
for i in [0, 45, 90]:
    s = ds[i]['observation.state']
    print(f"  sample[{i}]: [{', '.join(f'{v:+.3f}' for v in s)}]")

print(f"\nSample action values:")
for i in [0, 45, 90]:
    a = ds[i]['action']
    print(f"  sample[{i}]: [{', '.join(f'{v:+.3f}' for v in a)}]")

---
## Step 2: Understanding the State Vector

The SO-101 arm has **6 degrees of freedom**. Let's visualize a full episode to see how each joint moves during a pick-and-place task.

In [ ]:
JOINT_NAMES = ['shoulder_pan', 'shoulder_lift', 'elbow_flex',
               'wrist_flex', 'wrist_roll', 'gripper']
JOINT_COLORS = [ACCENT, BLUE, TEAL, PURPLE, WARM, '#555555']

# Extract one full episode
episode_0_states = []
episode_0_actions = []
for i in range(len(ds)):
    sample = ds[i]
    if sample['episode_index'] == 0:
        episode_0_states.append(sample['observation.state'])
        episode_0_actions.append(sample['action'])
    elif sample['episode_index'] > 0:
        break

states = np.array(episode_0_states)   # [T, 6]
actions = np.array(episode_0_actions)  # [T, 6]
T = len(states)
timesteps = np.arange(T)

print(f"Episode 0: {T} timesteps, 6 joints")
print(f"States shape:  {states.shape}")
print(f"Actions shape: {actions.shape}")

In [ ]:
# Plot joint angle trajectories: state vs action
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

for j in range(6):
    ax = axes[j]
    ax.plot(timesteps, states[:, j], color=JOINT_COLORS[j], linewidth=2,
            label='State (current)', alpha=0.9)
    ax.plot(timesteps, actions[:, j], color=JOINT_COLORS[j], linewidth=1.5,
            linestyle='--', alpha=0.6, label='Action (target)')
    ax.set_title(JOINT_NAMES[j], fontsize=12, fontweight='bold', color=JOINT_COLORS[j])
    ax.set_xlabel('Timestep', fontsize=10)
    ax.set_ylabel('Angle (rad)', fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

fig.suptitle('Episode 0 — Joint Trajectories (State vs Action)',
             fontsize=15, fontweight='bold', color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

print("Actions are slightly AHEAD of states — the policy predicts where to go next.")
print("The robot's state follows the actions with a small lag.")

In [ ]:
# Gripper open/close visualization
gripper_state = states[:, 5]
gripper_threshold = np.median(gripper_state)  # adaptive threshold
gripper_binary = (gripper_state > gripper_threshold).astype(float)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# Raw gripper value
axes[0].plot(timesteps, gripper_state, color=ACCENT, linewidth=2)
axes[0].axhline(y=gripper_threshold, color='gray', linestyle=':', linewidth=1, label=f'Threshold={gripper_threshold:.2f}')
axes[0].set_ylabel('Gripper Value', fontsize=11)
axes[0].set_title('Gripper — Raw Signal', fontsize=13, fontweight='bold', color=ACCENT)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Binary open/close
axes[1].fill_between(timesteps, gripper_binary, alpha=0.4, color=TEAL, step='mid')
axes[1].step(timesteps, gripper_binary, color=TEAL, linewidth=2, where='mid')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Closed', 'Open'], fontsize=11)
axes[1].set_xlabel('Timestep', fontsize=11)
axes[1].set_title('Gripper — Thresholded (Open/Closed)', fontsize=13, fontweight='bold', color=TEAL)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Count transitions
transitions = np.sum(np.abs(np.diff(gripper_binary)))
print(f"Gripper transitions (open<->close): {int(transitions)}")
print(f"This tells us when the robot grasps and releases the object.")

---
## Step 3: Camera Views and Preprocessing

SmolVLA uses **two camera views**: an overhead (workspace) camera and a wrist camera. Since we can't easily stream images in Colab, we'll simulate the views and demonstrate the preprocessing pipeline.

In [ ]:
from PIL import Image

def create_overhead_view(width=640, height=480):
    """Simulate an overhead camera view with colored bowls."""
    fig, ax = plt.subplots(figsize=(width/100, height/100), dpi=100)
    ax.set_xlim(0, width)
    ax.set_ylim(0, height)
    ax.set_facecolor('#E8E0D4')  # table color

    # Draw table border
    table = mpatches.FancyBboxPatch((20, 20), width-40, height-40,
                                    boxstyle='round,pad=10', facecolor='#D4C8B8',
                                    edgecolor='#A09080', linewidth=3)
    ax.add_patch(table)

    # Draw bowls
    bowl_positions = [(160, 350), (320, 350), (480, 350)]
    bowl_colors_rgb = ['#E05555', '#55B855', '#5585E0']
    bowl_labels = ['Red Bowl', 'Green Bowl', 'Blue Bowl']
    for (x, y), color, label in zip(bowl_positions, bowl_colors_rgb, bowl_labels):
        bowl = mpatches.Ellipse((x, y), 100, 60, facecolor=color, edgecolor='#333',
                                linewidth=2, alpha=0.8)
        ax.add_patch(bowl)
        ax.text(x, y-50, label, ha='center', fontsize=9, fontweight='bold', color='#333')

    # Draw box (the object to pick up)
    box = mpatches.FancyBboxPatch((280, 130), 80, 60,
                                  boxstyle='round,pad=3', facecolor=WARM,
                                  edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(320, 160, 'BOX', ha='center', va='center', fontsize=10,
            fontweight='bold', color='white')

    # Draw robot arm base
    arm_base = mpatches.Circle((320, 50), 30, facecolor='#888', edgecolor='#333',
                               linewidth=2, alpha=0.7)
    ax.add_patch(arm_base)
    ax.text(320, 50, 'ARM', ha='center', va='center', fontsize=8, fontweight='bold', color='white')

    ax.set_title('Overhead Camera (640x480)', fontsize=12, fontweight='bold', color=BLUE)
    ax.axis('off')
    plt.close(fig)
    return fig


def create_wrist_view(width=640, height=480):
    """Simulate a wrist-mounted camera view."""
    fig, ax = plt.subplots(figsize=(width/100, height/100), dpi=100)
    ax.set_xlim(0, width)
    ax.set_ylim(0, height)
    ax.set_facecolor('#D4C8B8')  # table surface close-up

    # Gripper fingers (from wrist camera perspective)
    left_finger = mpatches.FancyBboxPatch((80, 0), 60, 480,
                                          boxstyle='round,pad=5', facecolor='#666',
                                          edgecolor='#333', linewidth=2, alpha=0.6)
    right_finger = mpatches.FancyBboxPatch((500, 0), 60, 480,
                                           boxstyle='round,pad=5', facecolor='#666',
                                           edgecolor='#333', linewidth=2, alpha=0.6)
    ax.add_patch(left_finger)
    ax.add_patch(right_finger)

    # Box visible between fingers
    box = mpatches.FancyBboxPatch((240, 160), 160, 160,
                                  boxstyle='round,pad=5', facecolor=WARM,
                                  edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(320, 240, 'BOX', ha='center', va='center', fontsize=14,
            fontweight='bold', color='white')

    ax.set_title('Wrist Camera (640x480)', fontsize=12, fontweight='bold', color=TEAL)
    ax.axis('off')
    plt.close(fig)
    return fig


# Show side-by-side camera views
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overhead view
ax = axes[0]
ax.set_xlim(0, 640)
ax.set_ylim(0, 480)
ax.set_facecolor('#E8E0D4')
table = mpatches.FancyBboxPatch((20, 20), 600, 440,
                                boxstyle='round,pad=10', facecolor='#D4C8B8',
                                edgecolor='#A09080', linewidth=3)
ax.add_patch(table)
bowl_positions = [(160, 350), (320, 350), (480, 350)]
bowl_colors_rgb = ['#E05555', '#55B855', '#5585E0']
bowl_labels = ['Red', 'Green', 'Blue']
for (x, y), color, label in zip(bowl_positions, bowl_colors_rgb, bowl_labels):
    bowl = mpatches.Ellipse((x, y), 100, 60, facecolor=color, edgecolor='#333',
                            linewidth=2, alpha=0.8)
    ax.add_patch(bowl)
    ax.text(x, y-45, label, ha='center', fontsize=9, fontweight='bold', color='#333')
box = mpatches.FancyBboxPatch((280, 130), 80, 60,
                              boxstyle='round,pad=3', facecolor=WARM,
                              edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(320, 160, 'BOX', ha='center', va='center', fontsize=10,
        fontweight='bold', color='white')
ax.set_title('Overhead Camera (640x480)', fontsize=13, fontweight='bold', color=BLUE)
ax.axis('off')

# Wrist view
ax = axes[1]
ax.set_xlim(0, 640)
ax.set_ylim(0, 480)
ax.set_facecolor('#D4C8B8')
left_finger = mpatches.FancyBboxPatch((80, 0), 60, 480,
                                      boxstyle='round,pad=5', facecolor='#666',
                                      edgecolor='#333', linewidth=2, alpha=0.6)
right_finger = mpatches.FancyBboxPatch((500, 0), 60, 480,
                                       boxstyle='round,pad=5', facecolor='#666',
                                       edgecolor='#333', linewidth=2, alpha=0.6)
ax.add_patch(left_finger)
ax.add_patch(right_finger)
box2 = mpatches.FancyBboxPatch((240, 160), 160, 160,
                               boxstyle='round,pad=5', facecolor=WARM,
                               edgecolor='#333', linewidth=2)
ax.add_patch(box2)
ax.text(320, 240, 'BOX', ha='center', va='center', fontsize=14,
        fontweight='bold', color='white')
ax.set_title('Wrist Camera (640x480)', fontsize=13, fontweight='bold', color=TEAL)
ax.axis('off')

plt.suptitle('Two Camera Views — What SmolVLA Sees', fontsize=15,
             fontweight='bold', color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate the resize operation
original_size = (640, 480)
smolvla_internal_size = (384, 384)  # SmolVLA's internal resolution

print("Camera Preprocessing Pipeline:")
print(f"  Raw capture:       {original_size[0]} x {original_size[1]} (4:3 aspect ratio)")
print(f"  SmolVLA internal:  {smolvla_internal_size[0]} x {smolvla_internal_size[1]} (square crop + resize)")
print()

# Simulate a dummy image resize
dummy_image = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
pil_img = Image.fromarray(dummy_image)
resized = pil_img.resize(smolvla_internal_size, Image.BILINEAR)
print(f"  Before resize: {np.array(pil_img).shape}")
print(f"  After resize:  {np.array(resized).shape}")
print()

# The rename map
print("="*60)
print("RENAME MAP — Critical for Deployment")
print("="*60)
print()
print("Your dataset has:")
print('  "observation.images.webcam"      (overhead camera)')
print('  "observation.images.arm_cam"     (wrist camera)')
print()
print("SmolVLA expects:")
print('  "observation.images.camera1"     (first camera)')
print('  "observation.images.camera2"     (second camera)')
print()

rename_map = {
    'observation.images.webcam':   'observation.images.camera1',
    'observation.images.arm_cam':  'observation.images.camera2',
}

print("rename_map = {")
for k, v in rename_map.items():
    print(f'    "{k}": "{v}",')
print("}")
print()
print("Without this rename map, SmolVLA won't find the camera images!")

---
## Step 4: Action Chunking Deep Dive

SmolVLA predicts **50 actions at once** (an "action chunk"). The key question: do you call the model every step, or do you use a queue? This has **massive** implications for both speed and trajectory quality.

In [ ]:
class ActionChunkSimulator:
    """Efficient action chunking with a deque queue."""
    def __init__(self, chunk_size=50):
        self.queue = deque(maxlen=chunk_size)
        self.chunk_size = chunk_size
        self.model_calls = 0

    def get_action(self, observation, model_fn):
        """Get next action. Only calls model when queue is empty."""
        if len(self.queue) == 0:
            chunk = model_fn(observation)  # returns [chunk_size, action_dim]
            self.queue.extend(chunk)
            self.model_calls += 1
        return self.queue.popleft()

    def reset(self):
        self.queue.clear()
        self.model_calls = 0

print("ActionChunkSimulator created.")
print(f"  Chunk size: 50 actions per model call")
print(f"  Queue type: collections.deque (O(1) popleft)")

In [ ]:
# Define a synthetic "model" that generates smooth action chunks
ACTION_DIM = 6
CHUNK_SIZE = 50

call_counter = [0]  # mutable counter for tracking

def fake_model(observation):
    """Simulate SmolVLA: takes an observation, returns [50, 6] action chunk."""
    call_counter[0] += 1
    t_start = call_counter[0] * CHUNK_SIZE
    t = np.linspace(t_start, t_start + CHUNK_SIZE, CHUNK_SIZE)
    chunk = np.zeros((CHUNK_SIZE, ACTION_DIM))
    for j in range(ACTION_DIM):
        freq = 0.05 + j * 0.02
        chunk[:, j] = np.sin(freq * t + j * 0.5) * (0.5 + j * 0.1)
    return chunk

# --- BAD inference: call model EVERY step ---
call_counter[0] = 0
N_STEPS = 100
bad_actions = []
bad_model_calls = 0
for step in range(N_STEPS):
    obs = np.zeros(6)  # dummy observation
    chunk = fake_model(obs)  # expensive call every step!
    bad_actions.append(chunk[0])  # only use first action, waste the rest
    bad_model_calls += 1
bad_actions = np.array(bad_actions)
total_bad_calls = bad_model_calls

# --- GOOD inference: use the deque queue ---
call_counter[0] = 0
chunker = ActionChunkSimulator(chunk_size=CHUNK_SIZE)
good_actions = []
for step in range(N_STEPS):
    obs = np.zeros(6)  # dummy observation
    action = chunker.get_action(obs, fake_model)
    good_actions.append(action)
good_actions = np.array(good_actions)
total_good_calls = chunker.model_calls

print(f"BAD  inference: {total_bad_calls:3d} model calls for {N_STEPS} steps")
print(f"GOOD inference: {total_good_calls:3d} model calls for {N_STEPS} steps")
print(f"\nSpeedup: {total_bad_calls / total_good_calls:.0f}x fewer model calls!")

In [ ]:
# Side-by-side trajectory comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BAD: jerky trajectory (each action comes from a different chunk start)
ax = axes[0]
for j in range(3):  # show first 3 joints
    ax.plot(bad_actions[:, j], color=JOINT_COLORS[j], linewidth=1.5,
            label=JOINT_NAMES[j], alpha=0.8)
ax.set_title('BAD: Model Called Every Step', fontsize=13, fontweight='bold', color='#CC3333')
ax.set_xlabel('Timestep', fontsize=11)
ax.set_ylabel('Action Value', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.text(50, ax.get_ylim()[1]*0.9, f'{total_bad_calls} model calls',
        fontsize=11, color='#CC3333', fontweight='bold', ha='center')

# GOOD: smooth trajectory (actions from contiguous chunks)
ax = axes[1]
for j in range(3):
    ax.plot(good_actions[:, j], color=JOINT_COLORS[j], linewidth=1.5,
            label=JOINT_NAMES[j], alpha=0.8)
# Mark where model was called
for call_step in range(0, N_STEPS, CHUNK_SIZE):
    ax.axvline(x=call_step, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.set_title('GOOD: Action Chunking with Deque', fontsize=13, fontweight='bold', color=TEAL)
ax.set_xlabel('Timestep', fontsize=11)
ax.set_ylabel('Action Value', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.text(50, ax.get_ylim()[1]*0.9, f'{total_good_calls} model calls',
        fontsize=11, color=TEAL, fontweight='bold', ha='center')

plt.suptitle('Action Chunking: Jerky vs Smooth Execution',
             fontsize=15, fontweight='bold', color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

# Compute jerk (3rd derivative) as smoothness metric
bad_jerk = np.mean(np.abs(np.diff(bad_actions, n=3, axis=0)))
good_jerk = np.mean(np.abs(np.diff(good_actions, n=3, axis=0)))
print(f"\nTrajectory jerk (lower = smoother):")
print(f"  BAD:  {bad_jerk:.4f}")
print(f"  GOOD: {good_jerk:.4f}")
print(f"  Smoothness improvement: {bad_jerk / good_jerk:.1f}x")

---
## Step 5: The Voice Control Pipeline

In a real deployment, voice commands arrive **asynchronously** — the robot might be mid-task when a new instruction comes in. We need thread-safe task switching.

In [ ]:
class TaskState:
    """Thread-safe task state for voice-controlled switching."""
    def __init__(self, initial_task=""):
        self._task = initial_task
        self._lock = threading.Lock()
        self._change_count = 0
        self._history = []  # (timestamp, task)
        if initial_task:
            self._history.append((0.0, initial_task))

    @property
    def task(self):
        with self._lock:
            return self._task

    @task.setter
    def task(self, new_task):
        with self._lock:
            if new_task != self._task:
                self._task = new_task
                self._change_count += 1

    def update(self, new_task, timestamp):
        """Update task with a timestamp for logging."""
        with self._lock:
            if new_task != self._task:
                self._task = new_task
                self._change_count += 1
                self._history.append((timestamp, new_task))
                return True  # changed
            return False

    @property
    def change_count(self):
        with self._lock:
            return self._change_count

    @property
    def history(self):
        with self._lock:
            return list(self._history)

print("TaskState class created (thread-safe with threading.Lock).")

In [ ]:
# Simulate voice commands arriving at different times
voice_commands = [
    (0.0,  "Pick up the box and place it in the red bowl"),
    (5.0,  "Pick up the box and place it in the blue bowl"),
    (12.0, "Pick up the box and place it in the green bowl"),
]

# Simulate a 20-second deployment run at 10 Hz
DURATION = 20.0  # seconds
CONTROL_HZ = 10  # Hz
total_steps = int(DURATION * CONTROL_HZ)

task_state = TaskState()
timeline = []  # (time, current_task)
voice_idx = 0

for step in range(total_steps):
    sim_time = step / CONTROL_HZ

    # Check if a voice command arrives at this time
    if voice_idx < len(voice_commands) and sim_time >= voice_commands[voice_idx][0]:
        t_cmd, cmd = voice_commands[voice_idx]
        changed = task_state.update(cmd, sim_time)
        if changed:
            print(f"  t={sim_time:5.1f}s  TASK CHANGED: \"{cmd}\"")
        voice_idx += 1

    timeline.append((sim_time, task_state.task))

print(f"\nTotal task changes: {task_state.change_count}")
print(f"Task history: {task_state.history}")

In [ ]:
# Visualize task switching timeline
color_map = {
    'red': '#E05555',
    'blue': '#5585E0',
    'green': '#55B855',
}

fig, ax = plt.subplots(figsize=(14, 3))

# Draw colored bands for each task period
history = task_state.history
for i, (t_start, task) in enumerate(history):
    t_end = history[i+1][0] if i+1 < len(history) else DURATION
    # Extract color from task string
    for color_name, color_hex in color_map.items():
        if color_name in task:
            ax.axvspan(t_start, t_end, alpha=0.3, color=color_hex)
            ax.text((t_start + t_end) / 2, 0.5,
                    f"{color_name.upper()} bowl",
                    ha='center', va='center', fontsize=12,
                    fontweight='bold', color=color_hex)
            break

# Mark command arrival times
for t_cmd, cmd in voice_commands:
    ax.axvline(x=t_cmd, color=ACCENT, linewidth=2, linestyle='--')
    ax.text(t_cmd, 1.05, f'Voice cmd\nt={t_cmd:.0f}s',
            ha='center', va='bottom', fontsize=9, color=ACCENT, fontweight='bold')

ax.set_xlim(0, DURATION)
ax.set_ylim(0, 1)
ax.set_yticks([])
ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_title('Voice-Controlled Task Switching Timeline',
             fontsize=14, fontweight='bold', color=ACCENT)
ax.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
plt.show()

---
## Step 6: Intent Classification from Scratch

Instead of brittle keyword matching ("if 'red' in sentence..."), we build a **semantic intent classifier** using sentence embeddings. This handles synonyms, paraphrases, and creative descriptions.

In [ ]:
from sentence_transformers import SentenceTransformer, util

class IntentClassifier:
    """Semantic intent classifier for bowl color recognition."""
    def __init__(self):
        print("Loading sentence-transformers model (all-MiniLM-L6-v2)...")
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.colors = ['red', 'green', 'blue']
        self.references = {
            'red':   ['red', 'red bowl', 'blood', 'fire', 'ruby', 'scarlet',
                      'crimson', 'cherry', 'rose'],
            'green': ['green', 'green bowl', 'grass', 'leaves', 'emerald', 'forest',
                      'lime', 'jade', 'mint'],
            'blue':  ['blue', 'blue bowl', 'sky', 'ocean', 'azure', 'cobalt',
                      'navy', 'sapphire', 'indigo'],
        }
        # Pre-encode all reference phrases
        self._ref_embeddings = {}
        for color, phrases in self.references.items():
            self._ref_embeddings[color] = self.model.encode(phrases, convert_to_tensor=True)
        print("Intent classifier ready.")

    def classify(self, instruction):
        """Classify an instruction into a bowl color.
        Returns (color, confidence_score).
        """
        emb = self.model.encode(instruction, convert_to_tensor=True)
        best_color, best_score = None, -1
        scores = {}
        for color in self.colors:
            sim = util.cos_sim(emb, self._ref_embeddings[color]).max().item()
            scores[color] = sim
            if sim > best_score:
                best_score = sim
                best_color = color
        return best_color, best_score, scores

classifier = IntentClassifier()

In [ ]:
# Test on 20+ phrases including semantic/creative descriptions
test_phrases = [
    # Direct references
    "put it in the red bowl",
    "place it in the green bowl",
    "move to the blue bowl",
    # Semantic / creative descriptions
    "put it in the grassy bowl",
    "the ocean-colored one",
    "the blood-red bowl",
    "place it in the emerald bowl",
    "move to the sky-colored bowl",
    "the crimson bowl",
    "the forest-colored bowl",
    "cobalt bowl",
    "the leafy one",
    "put it where the ruby is",
    # More creative
    "the fire-colored bowl",
    "the bowl that looks like the sea",
    "the one that reminds me of grass",
    "cherry colored container",
    "the jade bowl",
    "sapphire dish",
    "mint green one",
    "the scarlet bowl",
    "navy colored bowl",
]

# Classify and print results
color_display = {'red': '\033[91mRED\033[0m', 'green': '\033[92mGREEN\033[0m', 'blue': '\033[94mBLUE\033[0m'}

print(f"{'Phrase':<42s} | {'Color':>8s} | {'Confidence':>10s}")
print("-" * 68)

results = []
for phrase in test_phrases:
    color, score, all_scores = classifier.classify(phrase)
    results.append((phrase, color, score, all_scores))
    print(f"{phrase:<42s} | {color:>8s} | {score:>10.3f}")

# Accuracy check (manual ground truth)
ground_truth = ['red', 'green', 'blue',
                'green', 'blue', 'red', 'green', 'blue', 'red', 'green',
                'blue', 'green', 'red', 'red', 'blue', 'green',
                'red', 'green', 'blue', 'green', 'red', 'blue']
correct = sum(1 for (_, c, _, _), gt in zip(results, ground_truth) if c == gt)
print(f"\nAccuracy: {correct}/{len(results)} ({100*correct/len(results):.0f}%)")

In [ ]:
# PCA visualization of embeddings
from sklearn.decomposition import PCA

# Collect all embeddings: test phrases + reference phrases
all_texts = []
all_labels = []  # color classification
all_types = []   # 'test' or 'reference'

# Reference phrases
for color, phrases in classifier.references.items():
    for p in phrases:
        all_texts.append(p)
        all_labels.append(color)
        all_types.append('reference')

# Test phrases
for phrase, color, score, _ in results:
    all_texts.append(phrase)
    all_labels.append(color)
    all_types.append('test')

# Encode all
all_embeddings = classifier.model.encode(all_texts, convert_to_tensor=True).cpu().numpy()

# PCA to 2D
pca = PCA(n_components=2)
coords = pca.fit_transform(all_embeddings)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

color_hex = {'red': '#E05555', 'green': '#55B855', 'blue': '#5585E0'}

for i, (text, label, ttype) in enumerate(zip(all_texts, all_labels, all_types)):
    marker = 'o' if ttype == 'reference' else '^'
    size = 80 if ttype == 'reference' else 120
    edge = 'black' if ttype == 'test' else 'none'
    ax.scatter(coords[i, 0], coords[i, 1], c=color_hex[label],
               marker=marker, s=size, edgecolors=edge, linewidths=1, alpha=0.8)
    if ttype == 'test':
        # Label only test phrases to avoid clutter
        short_text = text[:25] + '...' if len(text) > 25 else text
        ax.annotate(short_text, (coords[i, 0], coords[i, 1]),
                    textcoords='offset points', xytext=(8, 4),
                    fontsize=7, alpha=0.8)

# Legend
legend_elements = [
    plt.scatter([], [], c='gray', marker='o', s=80, label='Reference phrase'),
    plt.scatter([], [], c='gray', marker='^', s=120, edgecolors='black', linewidths=1, label='Test phrase'),
    mpatches.Patch(facecolor='#E05555', label='Red cluster'),
    mpatches.Patch(facecolor='#55B855', label='Green cluster'),
    mpatches.Patch(facecolor='#5585E0', label='Blue cluster'),
]
ax.legend(handles=legend_elements, fontsize=10, loc='best')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
ax.set_title('Intent Classifier — PCA of Sentence Embeddings',
             fontsize=14, fontweight='bold', color=ACCENT)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")
print("Test phrases (triangles) cluster near their reference phrases (circles).")

---
## Step 7: The Training Pipeline (Simulated)

In a real deployment, you would fine-tune SmolVLA on your own dataset. Here's what that looks like.

### Training Command

```bash
lerobot-train \
    --policy.path=lerobot/smolvla_base \
    --dataset.repo_id=RajatDandekar/so101_box_to_bowl \
    --dataset.episodes=[0,1,2,3,4,5,6,7,8,9] \
    --batch_size=64 \
    --steps=30000 \
    --eval.use_async_envs=false \
    --policy.rename_map='{"observation.images.webcam": "observation.images.camera1", "observation.images.arm_cam": "observation.images.camera2"}'
```

### Key Configuration Notes

| Flag | Purpose | Value |
|------|---------|-------|
| `--policy.path` | Base model to fine-tune | `lerobot/smolvla_base` |
| `--dataset.repo_id` | HuggingFace dataset | `RajatDandekar/so101_box_to_bowl` |
| `--batch_size` | Batch size (limited by VRAM) | 64 (A100), 32 (4090), 16 (T4) |
| `--steps` | Total training steps | 30,000 |
| `--policy.rename_map` | Map dataset camera names to model names | See above |

### GPU Options

| GPU | VRAM | Max Batch Size | Estimated Time (30K steps) |
|-----|------|----------------|----------------------------|
| A100 80GB | 80 GB | 64 | ~4 hours |
| A40 48GB | 48 GB | 48 | ~6 hours |
| RTX 4090 | 24 GB | 32 | ~8 hours |
| T4 (Colab free) | 16 GB | 8 | ~24 hours |

In [ ]:
# Simulate a training curve
np.random.seed(42)
total_steps = 30000
step_range = np.arange(1, total_steps + 1)

# Smooth exponential decay + noise
base_loss = 0.5 * np.exp(-step_range / 6000) + 0.05
noise = np.random.randn(total_steps) * 0.015 * np.exp(-step_range / 15000)
training_loss = base_loss + noise
training_loss = np.clip(training_loss, 0.02, 0.6)

# Smoothed version (moving average)
window = 500
smoothed_loss = np.convolve(training_loss, np.ones(window)/window, mode='valid')
smoothed_steps = step_range[window-1:]

fig, ax = plt.subplots(figsize=(12, 5))

# Raw loss (semi-transparent)
ax.plot(step_range[::10], training_loss[::10], color=BLUE, alpha=0.15, linewidth=0.5)
# Smoothed loss
ax.plot(smoothed_steps[::10], smoothed_loss[::10], color=ACCENT, linewidth=2.5,
        label=f'Smoothed (window={window})')

# Milestones
milestones = [(5000, 'Learns grasping'), (15000, 'Learns placement'), (25000, 'Fine-tunes colors')]
for step, label in milestones:
    loss_val = smoothed_loss[step - window] if step >= window else training_loss[step]
    ax.annotate(label, (step, loss_val),
                textcoords='offset points', xytext=(15, 15),
                fontsize=9, fontweight='bold', color=TEAL,
                arrowprops=dict(arrowstyle='->', color=TEAL, lw=1.5))

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Simulated SmolVLA Training Curve (30K Steps)',
             fontsize=14, fontweight='bold', color=ACCENT)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, total_steps)
ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.show()

print(f"Initial loss: {training_loss[0]:.3f}")
print(f"Final loss:   {training_loss[-1]:.3f}")
print(f"Loss reduction: {(1 - training_loss[-1]/training_loss[0])*100:.0f}%")

---
## Step 8: predict_action vs sample_actions

This is the **single most common deployment mistake**. Let's make the difference crystal clear.

In [ ]:
def fake_model_smooth(observation, chunk_size=50, action_dim=6):
    """Simulate SmolVLA generating a smooth 50-step action chunk."""
    t = np.linspace(0, 2 * np.pi, chunk_size)
    chunk = np.zeros((chunk_size, action_dim))
    # Each call produces a smooth trajectory segment
    base_phase = hash(str(observation)) % 100 * 0.1  # deterministic from observation
    for j in range(action_dim):
        chunk[:, j] = np.sin(t * (1 + j * 0.3) + base_phase + j) * (0.5 + j * 0.1)
    return chunk


def bad_inference(model_fn, observations, n_steps=100):
    """BAD: calls model every step, only uses first action from chunk."""
    actions = []
    model_calls = 0
    for i in range(n_steps):
        obs = observations[i] if i < len(observations) else observations[-1]
        chunk = model_fn(obs)   # expensive! 50 actions generated
        actions.append(chunk[0])  # only use first action, waste the rest
        model_calls += 1
    return np.array(actions), model_calls


def good_inference(model_fn, observations, n_steps=100):
    """GOOD: uses action queue, calls model only when queue is empty."""
    actions = []
    queue = deque()
    model_calls = 0
    call_steps = []  # track when model was called
    for i in range(n_steps):
        obs = observations[i] if i < len(observations) else observations[-1]
        if len(queue) == 0:
            chunk = model_fn(obs)
            queue.extend(chunk)
            model_calls += 1
            call_steps.append(i)
        actions.append(queue.popleft())
    return np.array(actions), model_calls, call_steps


# Generate dummy observations
N_STEPS = 100
dummy_obs = [np.random.randn(6) * 0.01 + i * 0.001 for i in range(N_STEPS)]

bad_actions, bad_calls = bad_inference(fake_model_smooth, dummy_obs, N_STEPS)
good_actions, good_calls, good_call_steps = good_inference(fake_model_smooth, dummy_obs, N_STEPS)

print(f"Results for {N_STEPS} steps:")
print(f"  BAD  (sample_actions every step): {bad_calls:3d} model calls")
print(f"  GOOD (predict_action with queue): {good_calls:3d} model calls")
print(f"  Model calls at steps: {good_call_steps}")

In [ ]:
# Bar chart: model calls comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart of model calls
ax = axes[0]
methods = ['sample_actions()\n(BAD)', 'predict_action()\n(GOOD)']
calls = [bad_calls, good_calls]
colors_bar = ['#CC3333', TEAL]
bars = ax.bar(methods, calls, color=colors_bar, edgecolor='white', linewidth=2, width=0.5)
for bar, val in zip(bars, calls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val}', ha='center', va='bottom', fontsize=16, fontweight='bold')
ax.set_ylabel('Model Calls', fontsize=12)
ax.set_title(f'Model Calls per {N_STEPS} Steps', fontsize=14, fontweight='bold', color=ACCENT)
ax.set_ylim(0, bad_calls * 1.15)
ax.grid(True, alpha=0.2, axis='y')

# Right: jerk comparison (trajectory smoothness)
ax = axes[1]
bad_jerk = np.mean(np.abs(np.diff(bad_actions, n=3, axis=0)), axis=0)
good_jerk = np.mean(np.abs(np.diff(good_actions, n=3, axis=0)), axis=0)

x = np.arange(6)
w = 0.35
ax.bar(x - w/2, bad_jerk, w, color='#CC3333', alpha=0.7, label='BAD (sample_actions)', edgecolor='white')
ax.bar(x + w/2, good_jerk, w, color=TEAL, alpha=0.7, label='GOOD (predict_action)', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(JOINT_NAMES, fontsize=9, rotation=30, ha='right')
ax.set_ylabel('Mean Jerk (3rd derivative)', fontsize=11)
ax.set_title('Trajectory Smoothness by Joint', fontsize=14, fontweight='bold', color=ACCENT)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

mean_bad_jerk = np.mean(bad_jerk)
mean_good_jerk = np.mean(good_jerk)
print(f"\nMean jerk across joints:")
print(f"  BAD:  {mean_bad_jerk:.4f}")
print(f"  GOOD: {mean_good_jerk:.4f}")
if mean_good_jerk > 0:
    print(f"  Smoothness ratio: {mean_bad_jerk / mean_good_jerk:.1f}x better with predict_action()")
else:
    print(f"  GOOD trajectory has near-zero jerk (perfectly smooth chunks)")

---
## Step 9: Full Pipeline Walkthrough + Summary

### End-to-End Deployment Pipeline

```
Voice Command
    |  ("pick up the box and place it in the red bowl")
    v
Intent Classifier (sentence-transformers)
    |  -> color = "red", confidence = 0.87
    v
TaskState (thread-safe)
    |  -> task = "Pick up the box and place it in the red bowl"
    v
Camera Capture (2x 640x480)
    |  -> overhead + wrist images
    v
Rename Map
    |  -> "observation.images.webcam" -> "observation.images.camera1"
    |  -> "observation.images.arm_cam" -> "observation.images.camera2"
    v
SmolVLA Model (predict_action)
    |  -> 50-step action chunk [50, 6]
    v
Action Queue (deque)
    |  -> popleft() one action per control step
    v
Robot Arm (SO-101)
    |  -> execute 6-DOF joint command
    v
Repeat at 10 Hz
```

In [ ]:
# Summary: DOs and DON'Ts
print("=" * 60)
print("DEPLOYMENT DOs and DON'Ts")
print("=" * 60)
print()

dos = [
    ("Use predict_action()",
     "Returns single action from internal queue. Efficient and smooth."),
    ("Use rename_map",
     "Map your camera names to what SmolVLA expects."),
    ("Use 640x480 cameras",
     "SmolVLA handles resizing internally. Don't resize yourself."),
    ("Use thread-safe TaskState",
     "Voice commands arrive on a different thread than the control loop."),
    ("Clear action queue on task change",
     "Old actions belong to the previous task."),
]

donts = [
    ("Call sample_actions() every step",
     "Wastes 49/50 actions. 50x slower and jerky trajectories."),
    ("Use sorted() for joint names",
     "Joint order must match the dataset. sorted() scrambles it."),
    ("Force 224x224 on cameras",
     "SmolVLA is NOT a standard ViT. It expects 640x480."),
    ("Forget the rename_map",
     "Model will silently receive blank images."),
    ("Block the control loop for voice processing",
     "Use a separate thread for voice/intent classification."),
]

print("  DOs:")
for title, desc in dos:
    print(f"    [+] {title}")
    print(f"        {desc}")

print()
print("  DON'Ts:")
for title, desc in donts:
    print(f"    [-] {title}")
    print(f"        {desc}")

In [ ]:
# Summary table
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)

summary = [
    ("Dataset",         "RajatDandekar/so101_box_to_bowl",    "45 episodes, 3 colors"),
    ("State vector",    "6-DOF joint angles",                 "shoulder, elbow, wrist, gripper"),
    ("Camera input",    "2x 640x480",                        "overhead + wrist"),
    ("Action chunk",    "50 steps per call",                  "Use deque for efficient queuing"),
    ("Voice control",   "Thread-safe TaskState",              "Intent classifier for colors"),
    ("Inference",       "predict_action()",                   "NOT sample_actions()"),
    ("Training",        "lerobot-train",                      "30K steps, batch_size=64"),
    ("Rename map",      "webcam->camera1, arm_cam->camera2", "Required for SmolVLA"),
]

print(f"{'Component':<20s} | {'Value':<35s} | {'Notes'}")
print("-" * 70)
for comp, val, notes in summary:
    print(f"{comp:<20s} | {val:<35s} | {notes}")

print("\nYou now understand every piece of the SmolVLA deployment pipeline!")

---
## Exercises

**1. Dataset Statistics**
Compute the per-joint mean and standard deviation across the entire dataset. Which joint has the highest variance? Print a table of `joint_name | mean | std` and identify the most variable joint.

**2. Chunk Overlap**
Modify the `ActionChunkSimulator` to only execute 15 of the 50 predicted actions before re-planning (i.e., discard the remaining 35 and request a fresh chunk). Compare the resulting trajectory smoothness (jerk) against the original full-chunk approach. When might partial execution be preferable?

**3. Adversarial Intent Phrases**
Test the `IntentClassifier` on adversarial negation phrases: "not the blue one", "anything but red", "every bowl except green". Does the classifier handle negation correctly? Why or why not? (Hint: think about what cosine similarity measures.)

**4. Add a Fourth Color**
Extend the `IntentClassifier` to support a 4th color: **yellow**. Add at least 6 reference phrases (e.g., "yellow", "gold", "sunshine", "lemon", "amber", "canary"). Test on 5 new yellow-related phrases and verify classification accuracy.

**5. Confidence Threshold**
Add a confidence threshold to the classifier: only accept a classification if `confidence > 0.5`. Otherwise, return `"unknown"` and print a clarification request. Test with ambiguous phrases like "the warm-colored bowl" or "the dark one" — do they fall below the threshold?

**6. Challenge: Full Mock Deployment**
Build a complete 60-second simulated deployment that combines everything:
- 3 voice command changes at t=5s, t=25s, and t=45s
- Robot joint trajectories responding to each task (use different target positions per color)
- Action chunk boundaries visible as vertical markers
- Task state shown as colored bands at the top
- Model call count displayed in the title

Plot everything on a single figure with 2 subplots: (top) task timeline with voice markers, (bottom) 6 joint trajectories with chunk boundaries.